In [11]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import urllib.parse
import os
from transformers import pipeline

def get_google_news(query, max_results=10):
    """Scrapes Google News via RSS to bypass bot protection."""
    print(f"Fetching news for: '{query}'...")
    encoded_query = urllib.parse.quote(query)
    url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-US&gl=US&ceid=US:en"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, features="xml")
    
    articles = soup.findAll('item')
    news_data = []
    
    for article in articles[:max_results]:
        news_data.append({
            'date_scraped': datetime.now().strftime("%Y-%m-%d"),
            'query': query,
            'title': article.title.text,
            'published': article.pubDate.text,
            'source': article.source.text
        })
        
    return news_data

def analyze_sentiment_finbert(news_data):
    """Scores headlines using FinBERT, a neural network trained on financial news."""
    print("Scoring headlines with FinBERT contextual AI...")
    
    # Load the pre-trained financial NLP model
    # (It will download a small model file the very first time you run it)
    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    
    for item in news_data:
        # FinBERT reads the whole context and returns a label: 'positive', 'negative', or 'neutral'
        result = finbert(item['title'])[0]
        label = result['label']
        score = result['score'] # Confidence from 0.0 to 1.0
        
        if label == 'positive':
            item['signal'] = 1
            item['confidence_score'] = round(score, 4)
        elif label == 'negative':
            item['signal'] = -1
            item['confidence_score'] = round(-score, 4) # Make bearish scores negative for your master_features
        else:
            item['signal'] = 0
            item['confidence_score'] = 0.0
            
    return news_data

def save_news_to_csv(data, filename="news_sentiment.csv"):
    df_new = pd.DataFrame(data)
    file_exists = os.path.isfile(filename)
    
    if file_exists:
        df_existing = pd.read_csv(filename)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        # Drop duplicates based on the EXACT headline title
        df_combined.drop_duplicates(subset=['title'], keep='last', inplace=True)
    else:
        df_combined = df_new
        
    df_combined.to_csv(filename, index=False)
    print(f"Successfully saved and deduplicated. {len(df_combined)} total unique headlines in {filename}")

if __name__ == "__main__":
    search_queries = [
        # Base pricing
        "WTI crude oil prices",
        "Florida gas prices",
        
        # Top 5 Iranian Fields from Wikipedia (Over 100 Billion Barrels combined)
        "Ahvaz oil field",
        "Marun oil field",
        "Aghajari oil field",
        "Gachsaran oil field",
        
        # The physical geographic chokepoints
        "Khuzestan province oil facility", # The province where all 5 fields are located
        "Strait of Hormuz tanker",         # Where the oil leaves the country
        "Gulf of Mexico refinery"          # Domestic weather/supply shocks
    ]
    
    all_news = []
    for query in search_queries:
        articles = get_google_news(query, max_results=5)
        all_news.extend(articles)
        
    # Process headlines through the new FinBERT NLP setup
    scored_news = analyze_sentiment_finbert(all_news)
    
    print("\n--- Top Scored Headlines ---")
    for item in scored_news[:5]:
        print(f"[{item['signal']}] (Score: {item['confidence_score']}) {item['title']}")
        
    save_news_to_csv(scored_news)

Fetching news for: 'WTI crude oil prices'...
Fetching news for: 'Florida gas prices'...
Fetching news for: 'Ahvaz oil field'...
Fetching news for: 'Marun oil field'...
Fetching news for: 'Aghajari oil field'...
Fetching news for: 'Gachsaran oil field'...
Fetching news for: 'Khuzestan province oil facility'...
Fetching news for: 'Strait of Hormuz tanker'...
Fetching news for: 'Gulf of Mexico refinery'...
Scoring headlines with FinBERT contextual AI...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Top Scored Headlines ---
[-1] (Score: -0.851) Dow tumbles more than 700 points as oil jumps, closing at new 2026 low under 47,000: Live updates - CNBC
[1] (Score: 0.8121) Brent Crude Futures Close Above $100 a Barrel For First Time Since 2022 - WSJ
[0] (Score: 0.0) Goldman Sachs raises Q4 Brent, WTI crude price forecast amid longer Hormuz disruption - Reuters
[-1] (Score: -0.8792) Oil prices volatile on conflicting reports about Strait of Hormuz - NBC News
[0] (Score: 0.0) 5 times oil prices surged about $100 — and how long they lasted - WGAL
Successfully saved and deduplicated. 68 total unique headlines in news_sentiment.csv
